# MATHS FOR BIG DATA

## Loads

In [11]:
import re
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

In [12]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arhip\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Data Loading

In [13]:
if os.path.exists("../data/train.csv"):
    df = pd.read_csv("../data/train.csv")
else:
    df = pd.read_csv("train.csv")

### Preprocessing (Entire Dataset)

In [4]:
df = df.dropna(subset=['text', 'class'])
stop_words = set(stopwords.words("catalan"))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r"[^\w\s]", '', text)
    tokens = word_tokenize(text, language='spanish')
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return tokens

df['tokens'] = df['text'].apply(preprocess)

## Word2Vec + Classifier Neural Network

### Word2Vec

In [5]:
train_df, test_df = train_test_split(df, test_size=0.3, random_state=123)

w2v_model = Word2Vec(sentences=train_df['tokens'], vector_size=250, window=5, min_count=3, workers=4, epochs=20)

def get_doc_vector(tokens):
    vecs = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if vecs:
        return np.mean(vecs, axis=0)
    else:
        return np.zeros(w2v_model.vector_size)

train_df['doc_vec'] = train_df['tokens'].apply(get_doc_vector)
test_df['doc_vec'] = test_df['tokens'].apply(get_doc_vector)

#### Data Preparation for the NN

In [6]:
X_train = np.vstack(train_df['doc_vec'].values)
X_test = np.vstack(test_df['doc_vec'].values)

le = LabelEncoder()
y_train = le.fit_transform(train_df['class'])
y_test = le.transform(test_df['class'])

num_classes = len(np.unique(y_train))

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

### Classifier Neural Network

In [7]:
class TextClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TextClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.dropout(out)
        return self.fc2(out)

### Predictions

In [8]:
model = TextClassifier(input_dim=250, hidden_dim=256, output_dim=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(30):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

model.eval()
correct, total = 0, 0
prediction_list = []

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        _, predicted = torch.max(preds, 1)
        prediction_list.extend(predicted.cpu().numpy())
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print(f"\nTest Accuracy: {correct / total:.4f}")

Epoch 1: Loss = 1290.8251
Epoch 2: Loss = 967.5132
Epoch 3: Loss = 895.8849
Epoch 4: Loss = 850.0089
Epoch 5: Loss = 816.5309
Epoch 6: Loss = 785.6617
Epoch 7: Loss = 768.3662
Epoch 8: Loss = 743.8308
Epoch 9: Loss = 729.2128
Epoch 10: Loss = 708.0526
Epoch 11: Loss = 700.9961
Epoch 12: Loss = 685.5477
Epoch 13: Loss = 673.1833
Epoch 14: Loss = 659.0224
Epoch 15: Loss = 648.3131
Epoch 16: Loss = 634.7579
Epoch 17: Loss = 637.2704
Epoch 18: Loss = 624.2441
Epoch 19: Loss = 621.8796
Epoch 20: Loss = 607.2957
Epoch 21: Loss = 602.5231
Epoch 22: Loss = 599.2364
Epoch 23: Loss = 585.1268
Epoch 24: Loss = 585.6683
Epoch 25: Loss = 573.1295
Epoch 26: Loss = 575.9564
Epoch 27: Loss = 556.9310
Epoch 28: Loss = 557.9889
Epoch 29: Loss = 552.1072
Epoch 30: Loss = 543.2624

Test Accuracy: 0.9519


### Prediction Writing

In [9]:
predicted_classes = le.inverse_transform(prediction_list)

output_df = pd.DataFrame({
    'id': test_df['id'].values,
    'class': predicted_classes,
})

output_df.to_csv('predictions.csv', index=False)